# Načtení a příprava dat

In [1]:
# nacteni vstupnich dat
# pickle je třeba nahrát do prostředí tohoto NB z:
# https://drive.google.com/file/d/15-sWqDXnhtDuVLyL1FentnL0XGuEzIP_/view?usp=drive_link

import pandas as pd
import pickle
with open("Data_lindat_zipformer_ft2_lm-extra06_allFeatures.pkl", "rb") as f:
    raw_data = pickle.load(f)

# rychle prohlednuti
raw_data

,kobar_kategorizace_definitivni,kobar_kategorizace_podle_RBANS,kobar_kategorizace_podle_ALBAV,kobar_kategorizace_podle_MAST,osoba_age,osoba_educationYears,osoba_sex,rbans_vyhodnoceni_souhrnny_skor,rbans_bezprostredni_pamet_test_uceni,rbans_bezprostredni_pamet_test_pameti,...,allFeatures_t10_tfidf_char_'ó',allFeatures_t10_tfidf_char_'ý',allFeatures_t10_tfidf_char_'č',allFeatures_t10_tfidf_char_'ň',allFeatures_t10_tfidf_char_'š',allFeatures_t10_mean_lev_f1_score,session_id,screening_id,splits,is_valid
0,0,0,0.0,0.0,60,20,1,121,34,21,...,NaN,NaN,NaN,NaN,NaN,NaN,i2W6KnmrYCBstJcR2YKNdm,scr-UFYw7EWtGytm93w4PcbnPY,train,True
1,0,0,0.0,0.0,75,18,0,126,33,16,...,NaN,NaN,NaN,NaN,NaN,NaN,dY2fr8RRPkSUoUdLYvYtbu,scr-NMnNHjnnGBrmwA7gWzJTNp,train,True
2,0,0,0.0,1.0,51,12,0,104,28,13,...,NaN,NaN,NaN,NaN,NaN,NaN,midcyThzXN6CUuKB5WmBZK,scr-Rbhnpx9Ms9qnCt43KkkSJB,train,True
3,0,0,0.0,1.0,69,19,1,104,26,20,...,NaN,NaN,NaN,NaN,NaN,NaN,C2iTC82ScZMsxfwSyT6FBy,scr-SPnEaj2zeULNDiCvLuaXhN,train,True
4,0,0,0.0,0.0,62,17,1,99,25,18,...,NaN,NaN,NaN,NaN,NaN,NaN,hM5GeZ9iUgs8f2FnfxxPHs,scr-LbyqVJeyu4zBfpmwz4hCKF,test,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
460,0,0,0.0,NaN,74,13,0,123,34,23,...,0.0,0.0,0.183155,0.0,0.0,0.7964,6a2fa3ee260933714ff61fac,scr-6a2fa2bf260933714ff61f94,train.extra,True
461,3,2,1.0,NaN,73,13,0,61,15,9,...,0.0,0.0,0.158877,0.0,0.0,0.6642,6a1d742a260933714ff400ec,scr-6a1d7480260933714ff40124,train.extra,True
462,0,0,0.0,NaN,76,20,1,122,26,15,...,0.0,0.0,0.208783,0.0,0.0,0.6880,6a157fa07d3e8142b3c1ff9f,scr-6a157e727d3e8142b3c1ff82,train.extra,True
463,0,0,0.0,NaN,53,12,0,119,30,19,...,0.0,0.0,0.120811,0.0,0.0,0.5688,6a146a147d3e8142b3c1e91e,scr-6a1457207d3e8142b3c1e6d2,train.extra,True


In [2]:
# kolik raw zaznamu mame...
raw_data.shape

(465, 3222)

In [3]:
# pocty v jednotlivych splitech, train extra idealne nechat pak na experimenty s objevovanim LLM features, resp. ty ktere maji jako diagnozu 0, 2, 3 (bez 1, ta bude vyfiltrovana nasledne)
raw_data['splits'].value_counts()

,count
splits,
train,294
train.extra,94
test,77


In [4]:
# distribuce diagnoz, nas budou zajimat vsechny krome hodnoty 1

diagnosis_distribution = raw_data["kobar_kategorizace_definitivni"].value_counts()
display(diagnosis_distribution)

,count
kobar_kategorizace_definitivni,
0,267
1,77
2,64
3,57


In [5]:
df_train = raw_data[raw_data["splits"].isin(["train", "train.extra"])]

for diag in [0, 1, 2, 3]:
    count = len(df_train[df_train["kobar_kategorizace_definitivni"] == diag])
    print(f"Diagnóza {diag}: {count}")

Diagnóza 0: 225
Diagnóza 1: 61
Diagnóza 2: 54
Diagnóza 3: 48


In [6]:
# jen inspekce...
raw_data.columns

Index(['kobar_kategorizace_definitivni', 'kobar_kategorizace_podle_RBANS',
       'kobar_kategorizace_podle_ALBAV', 'kobar_kategorizace_podle_MAST',
       'osoba_age', 'osoba_educationYears', 'osoba_sex',
       'rbans_vyhodnoceni_souhrnny_skor',
       'rbans_bezprostredni_pamet_test_uceni',
       'rbans_bezprostredni_pamet_test_pameti',
       ...
       'allFeatures_t10_tfidf_char_'ó'', 'allFeatures_t10_tfidf_char_'ý'',
       'allFeatures_t10_tfidf_char_'č'', 'allFeatures_t10_tfidf_char_'ň'',
       'allFeatures_t10_tfidf_char_'š'', 'allFeatures_t10_mean_lev_f1_score',
       'session_id', 'screening_id', 'splits', 'is_valid'],
      dtype='object', length=3222)

In [7]:
# filtr diagnozy: (0, 2, 3)
filtered_data = raw_data[raw_data["kobar_kategorizace_definitivni"].isin([0, 2, 3])].copy()

In [8]:
# binarizace 'new_diagnosis_label' pres lambda funkci
filtered_data['new_diagnosis_label'] = filtered_data["kobar_kategorizace_definitivni"].apply(lambda x: 1 if x in [2, 3] else 0)

In [9]:
# KLICOVA DATA PRO NAS: new_diag, task4 (popis obrazku), splits, nic dalsiho nechceme zpracovavat...
final_df = filtered_data[['new_diagnosis_label', 'task4_Complex scene description_obraz u jezera_recognized', 'splits']].copy()
display(final_df.head())

,new_diagnosis_label,task4_Complex scene description_obraz u jezera_recognized,splits
0,0,Kachna s kachňaty plave. Slečna si čte knížku ...,train
1,0,Žena se leží v lehátku a čte knihu s červeným ...,train
2,0,Pán loví ryby. Pes honí veverku. Děti hází míč...,train
3,0,Dáma v lehátku čte knihu. Děti si hrajou s míč...,train
4,0,Kluk si háže míč s holčičkou. Letadlo letí na ...,test


In [10]:
# rozdeleni na train a test df, inspekce

train_df = final_df[final_df['splits'].isin(['train', 'train.extra'])].drop(columns=['splits'])
test_df = final_df[final_df['splits'] == 'test'].drop(columns=['splits'])

print("Train DataFrame head:")
display(train_df.head())
print("\nTest DataFrame head:")
display(test_df.head())

Train DataFrame head:


,new_diagnosis_label,task4_Complex scene description_obraz u jezera_recognized
0,0,Kachna s kachňaty plave. Slečna si čte knížku ...
1,0,Žena se leží v lehátku a čte knihu s červeným ...
2,0,Pán loví ryby. Pes honí veverku. Děti hází míč...
3,0,Dáma v lehátku čte knihu. Děti si hrajou s míč...
5,0,Takže chlapeček s holčičkou si hrajou s balóne...



Test DataFrame head:


,new_diagnosis_label,task4_Complex scene description_obraz u jezera_recognized
4,0,Kluk si háže míč s holčičkou. Letadlo letí na ...
13,0,"V pozadí si hrajou děti, házejí si míčkem, běh..."
31,1,Tak paní leží pod slunečníkem. Čte knihu. Tady...
32,0,"Pes honí veverku, chlapec a dívka si hází míče..."
33,0,Paní si čte na lehátku knížku. Je pod sluneční...


# ML klasifikační část

In [11]:
# importy a priprava
# transformers je zamerne omezen na radu 4.x, aby se Colab beh neopiral o pripadne zmeny API v hlavni verzi 5.x
!pip install -q "transformers>=4.45,<5" datasets scikit-learn

import gc
import os
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
)
from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [12]:
# parametry modelu, trenovani, ...
# Zakladni hyperparametry jsou zamerne shodne s referencnim notebookem.

MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 8
LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.1

# Pro reproducibilitu
SEED = 1704

# Vsechny modely pouzivaji standardni Hugging Face rozhrani AutoTokenizer + AutoModelForSequenceClassification.
# Small-E-Czech je ELECTRA a lze jej proto trenovat stejnym kodem jako ostatni modely.
MODEL_CONFIGS = [
    {
        "key": "fernet_c5_roberta",
        "display_name": "C5-FERNET (FERNET-C5-RoBERTa)",
        "model_name": "fav-kky/FERNET-C5-RoBERTa",
    },
    {
        "key": "robeczech_base",
        "display_name": "RobeCzech base",
        "model_name": "ufal/robeczech-base",
    },
    {
        "key": "czert_b_base_cased",
        "display_name": "CZERT-B base cased",
        "model_name": "UWB-AIR/Czert-B-base-cased",
    },
    {
        "key": "mbert_base_cased",
        "display_name": "mBERT base multilingual cased",
        "model_name": "google-bert/bert-base-multilingual-cased",
    },
    {
        "key": "small_e_czech",
        "display_name": "Small-E-Czech",
        "model_name": "Seznam/small-e-czech",
    },
]

# Pro rychlejsi ladeni lze seznam zuzit, napr.:
# MODEL_CONFIGS = [MODEL_CONFIGS[0], MODEL_CONFIGS[1]]


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_global_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Použité zařízení:", device)
if device.type != "cuda":
    print("UPOZORNĚNÍ: Notebook je určen primárně pro Google Colab s GPU; na CPU bude trénování velmi pomalé.")


Použité zařízení: cuda


## Kontrola modelů před trénováním

Všechny checkpointy jsou před spuštěním experimentů ověřeny přes `AutoConfig`. U základních předtrénovaných modelů je při načtení `AutoModelForSequenceClassification(..., num_labels=2)` **očekávané**, že se nová klasifikační hlava inicializuje náhodně; následně se učí na našich datech.

Použité checkpointy:

- `fav-kky/FERNET-C5-RoBERTa`
- `ufal/robeczech-base`
- `UWB-AIR/Czert-B-base-cased`
- `google-bert/bert-base-multilingual-cased`
- `Seznam/small-e-czech`


In [13]:
# Rychla kontrola, ze vsechny modelove konfigurace existuji a podporuji zvolenou delku vstupu.
for model_cfg in MODEL_CONFIGS:
    hf_cfg = AutoConfig.from_pretrained(model_cfg["model_name"])
    max_positions = getattr(hf_cfg, "max_position_embeddings", None)

    print(
        f"{model_cfg['display_name']}: "
        f"model_type={hf_cfg.model_type}, "
        f"max_position_embeddings={max_positions}"
    )

    if max_positions is not None and MAX_LENGTH > max_positions:
        raise ValueError(
            f"MAX_LENGTH={MAX_LENGTH} je pro {model_cfg['model_name']} vetsi nez "
            f"max_position_embeddings={max_positions}."
        )


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/637 [00:00<?, ?B/s]

C5-FERNET (FERNET-C5-RoBERTa): model_type=roberta, max_position_embeddings=514


config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

RobeCzech base: model_type=roberta, max_position_embeddings=514


...

CZERT-B base cased: model_type=bert, max_position_embeddings=512


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

mBERT base multilingual cased: model_type=bert, max_position_embeddings=512


config.json:   0%|          | 0.00/410 [00:00<?, ?B/s]

Small-E-Czech: model_type=electra, max_position_embeddings=512


In [14]:
# Klicove sloupce: new_diagnosis_label, task4_Complex scene description_obraz u jezera_recognized

TEXT_COLUMN = "task4_Complex scene description_obraz u jezera_recognized"
LABEL_COLUMN = "new_diagnosis_label"
required_columns = {TEXT_COLUMN, LABEL_COLUMN}

if not required_columns.issubset(train_df.columns):
    raise ValueError(f"Trénovací data musí obsahovat sloupce: {required_columns}")
if not required_columns.issubset(test_df.columns):
    raise ValueError(f"Testovací data musí obsahovat sloupce: {required_columns}")

# Odstraneni radku s NaN v textu / labelu; provede se jednou, shodne pro vsechny modely.
train_df = train_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).copy()
test_df = test_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).copy()

# Prevod na spravne typy
train_df[TEXT_COLUMN] = train_df[TEXT_COLUMN].astype(str)
train_df[LABEL_COLUMN] = train_df[LABEL_COLUMN].astype(int)
test_df[TEXT_COLUMN] = test_df[TEXT_COLUMN].astype(str)
test_df[LABEL_COLUMN] = test_df[LABEL_COLUMN].astype(int)

texts_train_all = train_df[TEXT_COLUMN].tolist()
labels_train_all = train_df[LABEL_COLUMN].to_numpy()
test_texts = test_df[TEXT_COLUMN].tolist()
test_labels = test_df[LABEL_COLUMN].to_numpy()

# Validace binarnich labelu pred trenovanim.
train_unique_labels = set(np.unique(labels_train_all).tolist())
test_unique_labels = set(np.unique(test_labels).tolist())
if not train_unique_labels.issubset({0, 1}) or not test_unique_labels.issubset({0, 1}):
    raise ValueError(
        f"Ocekavany binarni label 0/1. Train={train_unique_labels}, test={test_unique_labels}"
    )

print("Počet instancí v trénovacích datech:", len(texts_train_all))
print("Počet instancí v testovacích datech:", len(test_texts))
print("Distribuce train labelů:", dict(zip(*np.unique(labels_train_all, return_counts=True))))
print("Distribuce test labelů:", dict(zip(*np.unique(test_labels, return_counts=True))))


Počet instancí v trénovacích datech: 327
Počet instancí v testovacích datech: 61
Distribuce train labelů: {np.int64(0): np.int64(225), np.int64(1): np.int64(102)}
Distribuce test labelů: {np.int64(0): np.int64(42), np.int64(1): np.int64(19)}


In [15]:
# Split to TRAIN/VAL -- vytvoren JEDNOU a pouzit beze zmeny pro vsechny modely.

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts_train_all,
    labels_train_all,
    test_size=0.25,
    random_state=SEED,
    stratify=labels_train_all,
)

print("Počet instancí v trénovací množině:", len(train_texts))
print("Počet instancí ve validační množině:", len(val_texts))
print("Train distribuce:", dict(zip(*np.unique(train_labels, return_counts=True))))
print("Val distribuce:", dict(zip(*np.unique(val_labels, return_counts=True))))


Počet instancí v trénovací množině: 245
Počet instancí ve validační množině: 82
Train distribuce: {np.int64(0): np.int64(169), np.int64(1): np.int64(76)}
Val distribuce: {np.int64(0): np.int64(56), np.int64(1): np.int64(26)}


In [16]:
# funkce na tokenizaci...
def tokenize_texts(text_list, tokenizer, max_length):
    """
    Tokenizuje seznam textu pomoci daneho tokenizeru.
    Vraci input_ids a attention_mask jako PyTorch tensory.

    token_type_ids nejsou pro tento experiment nutne:
    vsechny vstupy jsou jednosekvencni a BERT/ELECTRA v takovem pripade standardne pouziji segment 0.
    """
    encodings = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    return encodings["input_ids"], encodings["attention_mask"]


def make_dataloader(texts, labels, tokenizer, batch_size, shuffle, seed):
    input_ids, attention_mask = tokenize_texts(texts, tokenizer, MAX_LENGTH)
    labels_tensor = torch.tensor(labels, dtype=torch.long)

    dataset = TensorDataset(input_ids, attention_mask, labels_tensor)

    # Generator fixuje poradi shuffle a tim zlepsuje reprodukovatelnost mezi behy.
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
    )


In [17]:
# Spolecna evaluacni funkce pro VAL i TEST.
def evaluate_model(model, dataloader, device, with_loss=True, desc="Evaluace"):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc):
            batch_input_ids = batch[0].to(device)
            batch_attention_mask = batch[1].to(device)
            batch_labels = batch[2].to(device)

            outputs = model(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
                labels=batch_labels if with_loss else None,
            )

            if with_loss:
                total_loss += outputs.loss.item()

            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)[:, 1]

            all_labels.extend(batch_labels.detach().cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

    all_labels = np.asarray(all_labels)
    all_probs = np.asarray(all_probs)
    preds = (all_probs >= 0.5).astype(int)

    accuracy = accuracy_score(all_labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        preds,
        average="binary",
        zero_division=0,
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float("nan")

    avg_loss = total_loss / len(dataloader) if with_loss and len(dataloader) > 0 else float("nan")

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": auc,
        "labels": all_labels,
        "probs": all_probs,
        "preds": preds,
    }


In [18]:
# Jedna kompletni experimentální smycka = nacteni modelu, tokenizace, trening, VAL po kazde epose, TEST a ulozeni predikci.
def run_model_experiment(model_cfg):
    model_name = model_cfg["model_name"]
    display_name = model_cfg["display_name"]
    model_key = model_cfg["key"]

    print("\n" + "=" * 100)
    print(f"MODEL: {display_name}")
    print(f"HF checkpoint: {model_name}")
    print("=" * 100)

    # Kazdy model zacina ze stejneho seedu.
    set_global_seed(SEED)

    # Tokenizer: fast varianta je preferovana; fallback udrzuje kompatibilitu i se starsimi checkpointy.
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    except Exception as exc:
        print(f"Fast tokenizer se nepodarilo nacist ({type(exc).__name__}); zkousim use_fast=False.")
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

    # Zakladni pretrained checkpoint nema klasifikacni hlavu pro nas task.
    # AutoModelForSequenceClassification ji vytvori s num_labels=2; varovani o nove inicializovanych vahach je ocekavane.
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label={0: "NEGATIVE", 1: "POSITIVE"},
        label2id={"NEGATIVE": 0, "POSITIVE": 1},
    )
    model.to(device)

    print("\nTokenizuji trénovací, validační a testovací data...")
    train_dataloader = make_dataloader(
        train_texts, train_labels, tokenizer, BATCH_SIZE, shuffle=True, seed=SEED
    )
    val_dataloader = make_dataloader(
        val_texts, val_labels, tokenizer, BATCH_SIZE, shuffle=False, seed=SEED
    )
    test_dataloader = make_dataloader(
        test_texts, test_labels, tokenizer, BATCH_SIZE, shuffle=False, seed=SEED
    )

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_dataloader) * EPOCHS
    num_warmup_steps = int(WARMUP_RATIO * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=total_steps,
    )

    history = []
    print("\n===== ZAČÁTEK TRÉNOVÁNÍ =====")

    for epoch in range(EPOCHS):
        print(f"\n--- Epocha {epoch + 1}/{EPOCHS} ---")
        model.train()
        total_train_loss = 0.0

        for batch in tqdm(train_dataloader, desc=f"Trénování [{display_name}]"):
            batch_input_ids = batch[0].to(device)
            batch_attention_mask = batch[1].to(device)
            batch_labels = batch[2].to(device)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
                labels=batch_labels,
            )

            loss = outputs.loss
            loss.backward()

            # Orezani gradientu stejne jako v referencnim notebooku.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_dataloader)
        print(f"Průměrná trénovací ztráta: {avg_train_loss:.4f}")

        # =======================
        # EVALUACE NA VAL SUBSETU
        # =======================
        val_metrics = evaluate_model(
            model,
            val_dataloader,
            device,
            with_loss=True,
            desc=f"Validace [{display_name}]",
        )

        print(f"Validační ztráta: {val_metrics['loss']:.4f}")
        print(f"Validační Accuracy:  {val_metrics['accuracy']:.4f}")
        print(f"Validační Precision: {val_metrics['precision']:.4f}")
        print(f"Validační Recall:    {val_metrics['recall']:.4f}")
        print(f"Validační F1:        {val_metrics['f1']:.4f}")
        print(f"Validační ROC AUC:   {val_metrics['roc_auc']:.4f}")

        history.append({
            "model_key": model_key,
            "model_name": model_name,
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_roc_auc": val_metrics["roc_auc"],
        })

    print("\n===== TRÉNOVÁNÍ DOKONČENO =====")

    # =========================================
    # TEST
    # =========================================
    print("\nPredikuji na testovacích datech...")
    test_metrics = evaluate_model(
        model,
        test_dataloader,
        device,
        with_loss=False,
        desc=f"Test predikce [{display_name}]",
    )

    print("\n===== METRIKY NA TESTOVACÍ MNOŽINĚ =====")
    print(f"Test Accuracy:  {test_metrics['accuracy']:.4f}")
    print(f"Test Precision: {test_metrics['precision']:.4f}")
    print(f"Test Recall:    {test_metrics['recall']:.4f}")
    print(f"Test F1:        {test_metrics['f1']:.4f}")
    print(f"Test ROC AUC:   {test_metrics['roc_auc']:.4f}")

    cm = confusion_matrix(test_metrics["labels"], test_metrics["preds"], labels=[0, 1])
    print("\n===== MATICE ZÁMĚN =====")
    print(cm)
    tn, fp, fn, tp = cm.ravel()

    # =========================================
    # Ulozeni predikci do CSV
    # =========================================
    output_test_csv_path = f"./texttest_predicted_{model_key}.csv"
    test_output_df = test_df.copy()
    test_output_df["Pravdepodobnost"] = test_metrics["probs"]
    test_output_df["Klasifikace_pred"] = test_metrics["preds"]
    test_output_df.to_csv(output_test_csv_path, index=False)
    print(f"\nVýsledné predikce byly uloženy do souboru:\n{output_test_csv_path}")

    # Posledni VAL epocha odpovida logice puvodniho notebooku (bez early stopping / vyberu best checkpointu).
    last_val = history[-1]
    summary = {
        "model_key": model_key,
        "display_name": display_name,
        "model_name": model_name,
        "val_accuracy": last_val["val_accuracy"],
        "val_precision": last_val["val_precision"],
        "val_recall": last_val["val_recall"],
        "val_f1": last_val["val_f1"],
        "val_roc_auc": last_val["val_roc_auc"],
        "test_accuracy": test_metrics["accuracy"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
        "test_roc_auc": test_metrics["roc_auc"],
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "prediction_csv": output_test_csv_path,
    }

    history_df = pd.DataFrame(history)

    # Uvolneni GPU/RAM pred dalsim modelem.
    del optimizer, scheduler, train_dataloader, val_dataloader, test_dataloader
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary, history_df


# Trénování všech klasifikátorů

Následující buňka spustí modely **sekvenčně**, nikoli současně. To je záměrné kvůli paměti GPU v Google Colab. Každý model dostane stejná data, stejný TRAIN/VAL split, stejný seed a stejné základní hyperparametry.


In [19]:
all_results = []
all_histories = {}

for model_cfg in MODEL_CONFIGS:
    summary, history_df = run_model_experiment(model_cfg)
    all_results.append(summary)
    all_histories[model_cfg["key"]] = history_df

print("\n\n===== VŠECHNY EXPERIMENTY DOKONČENY =====")



MODEL: C5-FERNET (FERNET-C5-RoBERTa)
HF checkpoint: fav-kky/FERNET-C5-RoBERTa


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Průměrná trénovací ztráta: 0.6139


Validace [Small-E-Czech]:   0%|          | 0/11 [00:00<?, ?it/s]

Validační ztráta: 0.6087
Validační Accuracy:  0.6829
Validační Precision: 0.0000
Validační Recall:    0.0000
Validační F1:        0.0000
Validační ROC AUC:   0.8180

===== TRÉNOVÁNÍ DOKONČENO =====

Predikuji na testovacích datech...


Test predikce [Small-E-Czech]:   0%|          | 0/8 [00:00<?, ?it/s]


===== METRIKY NA TESTOVACÍ MNOŽINĚ =====
Test Accuracy:  0.6885
Test Precision: 0.0000
Test Recall:    0.0000
Test F1:        0.0000
Test ROC AUC:   0.7820

===== MATICE ZÁMĚN =====
[[42  0]
 [19  0]]

Výsledné predikce byly uloženy do souboru:
./texttest_predicted_small_e_czech.csv


===== VŠECHNY EXPERIMENTY DOKONČENY =====


# Souhrnné srovnání


In [20]:
results_df = pd.DataFrame(all_results)

# Sloupce v poradi vhodnem pro rychle srovnani.
comparison_columns = [
    "display_name",
    "model_name",
    "val_accuracy",
    "val_precision",
    "val_recall",
    "val_f1",
    "val_roc_auc",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_roc_auc",
    "tn",
    "fp",
    "fn",
    "tp",
    "prediction_csv",
]

results_df = results_df[comparison_columns]
# Poradi nechavame podle MODEL_CONFIGS. TEST metriky slouzi k finalnimu reportingu, ne k vyberu modelu.
results_df = results_df.reset_index(drop=True)

display(results_df)

RESULTS_CSV_PATH = "./model_comparison_metrics.csv"
results_df.to_csv(RESULTS_CSV_PATH, index=False)
print(f"Souhrnné metriky byly uloženy do: {RESULTS_CSV_PATH}")


,display_name,model_name,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,tn,fp,fn,tp,prediction_csv
0,C5-FERNET (FERNET-C5-RoBERTa),fav-kky/FERNET-C5-RoBERTa,0.780488,0.700000,0.538462,0.608696,0.837912,0.786885,0.714286,0.526316,0.606061,0.744361,38,4,9,10,./texttest_predicted_fernet_c5_roberta.csv
1,RobeCzech base,ufal/robeczech-base,0.780488,0.750000,0.461538,0.571429,0.828297,0.868852,1.000000,0.578947,0.733333,0.835840,42,0,8,11,./texttest_predicted_robeczech_base.csv
2,CZERT-B base cased,UWB-AIR/Czert-B-base-cased,0.792683,0.764706,0.500000,0.604651,0.848214,0.819672,0.700000,0.736842,0.717949,0.822055,36,6,5,14,./texttest_predicted_czert_b_base_cased.csv
3,mBERT base multilingual cased,google-bert/bert-base-multilingual-cased,0.768293,0.733333,0.423077,0.536585,0.776786,0.754098,0.611111,0.578947,0.594595,0.798246,35,7,8,11,./texttest_predicted_mbert_base_cased.csv
4,Small-E-Czech,Seznam/small-e-czech,0.682927,0.000000,0.000000,0.000000,0.817995,0.688525,0.000000,0.000000,0.000000,0.781955,42,0,19,0,./texttest_predicted_small_e_czech.csv


Souhrnné metriky byly uloženy do: ./model_comparison_metrics.csv


In [ ]:
# Volitelna inspekce prubehu validacnich metrik po epochach pro jednotlive modely.
# Napr. all_histories["robeczech_base"]

for model_key, history_df in all_histories.items():
    print(f"\n===== HISTORY: {model_key} =====")
    display(history_df)


# Volitelné uložení modelu/tokenizeru

Stejně jako v referenčním notebooku není ukládání vah aktivní automaticky. Pokud chcete ukládat fine-tuned checkpointy, je vhodné to provést uvnitř `run_model_experiment` ještě před `del model, tokenizer`, například přes `model.save_pretrained(...)` a `tokenizer.save_pretrained(...)`.
